In [0]:
df_raw = spark.read.text("/Volumes/gharchive/bronze_dev/landing_files/2020-06-08-*.json.gz")

In [0]:
df_raw.show(5, truncate=False)

read.text instead of read.json because the payload structure differs by event type, and letting Spark infer one schema across mixed event types silently drops or nulls fields that don't fit."

In [0]:
from pyspark.sql.functions import get_json_object

df_test = df_raw.select(
    get_json_object("value", "$.id").alias("event_id"),
    get_json_object("value", "$.type").alias("event_type"),
    get_json_object("value", "$.actor.id").alias("actor_id"),
    get_json_object("value", "$.actor.login").alias("actor_login"),
    get_json_object("value", "$.repo.id").alias("repo_id"),
    get_json_object("value", "$.repo.name").alias("repo_name"),
    get_json_object("value", "$.payload").alias("payload"),
    get_json_object("value", "$.created_at").alias("created_at"),
    get_json_object("value", "$.org.id").alias("org_id"),
    get_json_object("value", "$.org.login").alias("org_login")
)

df_test.show(5, truncate=False)


In [0]:
df_test.printSchema()

In [0]:
df_test.selectExpr(
    "count(*) as total_rows",
    "count(org_id) as non_null_org_id",
    "count(org_login) as non_null_org_login"
).show()

In [0]:
from pyspark.sql.functions import to_date, col

df_bronze = df_test.withColumn("actor_id", col("actor_id").cast("long")) \
                   .withColumn("repo_id", col("repo_id").cast("long")) \
                    .withColumn("org_id", col("org_id").cast("long")) \
                    .withColumn("event_date", to_date(col("created_at")))


df_bronze.show(5)


In [0]:
df_bronze.printSchema()

In [0]:
df_bronze.select("created_at", "event_date").show(5, truncate=False)

In [0]:
df_bronze.selectExpr("count(*) as total_rows", "count(event_date) as non_null_dates").show()

In [0]:
df_bronze.write\
         .format("delta")\
         .mode("overwrite")\
         .option("replaceWhere", "event_date = '2020-06-08'")\
         .partitionBy("event_date")\
         .saveAsTable("gharchive.bronze_dev.events")

In [0]:
count_before = spark.table("gharchive.bronze_dev.events").count()
distinct_before = spark.table("gharchive.bronze_dev.events") \
    .select("event_id").distinct().count()

print("Row count before rerun:", count_before)
print("Distinct event_id before rerun:", distinct_before)

In [0]:
count_after = spark.table("gharchive.bronze_dev.events").count()
distinct_after = spark.table("gharchive.bronze_dev.events") \
    .select("event_id").distinct().count()

print("Row count after rerun:", count_after)
print("Distinct event_id after rerun:", distinct_after)

In [0]:
print("Row count match:", count_before == count_after)
print("Distinct event_id match:", distinct_before == distinct_after)

In [0]:
from pyspark.sql.functions import get_json_object
from pyspark.sql.functions import to_date, col

dates = ["2020-06-08", "2020-06-09", "2020-06-10", "2020-06-11",
         "2020-06-12", "2020-06-13", "2020-06-14"]

for d in dates:
    df_raw_final = spark.read.text(f"/Volumes/gharchive/bronze_dev/landing_files/{d}-*.json.gz")

    df_extract_final = df_raw_final.select(
        get_json_object("value", "$.id").alias("event_id"),
        get_json_object("value", "$.type").alias("event_type"),
        get_json_object("value", "$.actor.id").alias("actor_id"),
        get_json_object("value", "$.actor.login").alias("actor_login"),
        get_json_object("value", "$.repo.id").alias("repo_id"),
        get_json_object("value", "$.repo.name").alias("repo_name"),
        get_json_object("value", "$.payload").alias("payload"),
        get_json_object("value", "$.created_at").alias("created_at"),
        get_json_object("value", "$.org.id").alias("org_id"),
        get_json_object("value", "$.org.login").alias("org_login")
    )

    df_bronze_final = df_extract_final.withColumn("actor_id", col("actor_id").cast("long")) \
        .withColumn("repo_id", col("repo_id").cast("long")) \
        .withColumn("org_id", col("org_id").cast("long")) \
        .withColumn("event_date", to_date(col("created_at")))

    df_bronze_final.write \
                   .format("delta") \
                   .mode("overwrite") \
                   .option("replaceWhere", f"event_date = '{d}'") \
                   .partitionBy("event_date") \
                   .saveAsTable("gharchive.bronze_dev.events")


    print(f"{d}: {df_bronze_final.count()} rows written")

In [0]:
for d in dates:
    files = dbutils.fs.ls(f"/Volumes/gharchive/bronze_dev/landing_files/")
    count = len([f for f in files if f.name.startswith(d)])
    print(f"{d}: {count} files")